# Model Training - Water Potability

Mô hình sử dụng:
- Logistic Regression
- Support Vector Machine (SVM)

Quy trình:
- Median Imputation
- StandardScaler
- Train/Test = 80/20
- 5-Fold Stratified Cross Validation
- Hyperparameter Tuning bằng GridSearchCV

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.exceptions import UndefinedMetricWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC


warnings.filterwarnings(
    "ignore",
    category=UndefinedMetricWarning
)


# Tìm dataset
possible_paths = [
    Path("../data/water_potability.csv"),
    Path("ai-models/data/water_potability.csv"),
    Path("../ai-models/data/water_potability.csv"),
]

data_path = next(
    (path for path in possible_paths if path.exists()),
    None
)

if data_path is None:
    raise FileNotFoundError(
        "Không tìm thấy water_potability.csv"
    )


# Xác định thư mục project
project_root = data_path.resolve().parents[2]

src_path = (
    project_root
    / "ai-models"
    / "src"
)

if str(src_path) not in sys.path:
    sys.path.insert(
        0,
        str(src_path)
    )


from preprocess import (
    load_dataset,
    split_features_target,
    split_train_test,
    build_scaled_preprocessor,
    RANDOM_STATE,
)


# Load dữ liệu
df = load_dataset(data_path)

X, y = split_features_target(df)

X_train, X_test, y_train, y_test = (
    split_train_test(X, y)
)


print("Dataset:", df.shape)

print(
    "Train:",
    X_train.shape
)

print(
    "Test:",
    X_test.shape
)

print(
    "Train/Test overlap:",
    len(
        X_train.index.intersection(
            X_test.index
        )
    )
)

Dataset: (3276, 10)
Train: (2620, 9)
Test: (656, 9)
Train/Test overlap: 0


## 1. Cross Validation Setup

In [ ]:
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)


scoring_metrics = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}


def cv_summary(results):

    rows = []

    for metric in scoring_metrics:

        values = results[
            f"test_{metric}"
        ]

        rows.append({
            "Metric": (
                metric.upper()
                if metric != "roc_auc"
                else "ROC_AUC"
            ),
            "Mean": np.mean(values),
            "Std": np.std(values),
        })

    return pd.DataFrame(rows)


def train_validation_summary(results):

    rows = []

    for metric in scoring_metrics:

        train_mean = np.mean(
            results[
                f"train_{metric}"
            ]
        )

        validation_mean = np.mean(
            results[
                f"test_{metric}"
            ]
        )

        rows.append({
            "Metric": (
                metric.upper()
                if metric != "roc_auc"
                else "ROC_AUC"
            ),

            "Train_Mean":
                train_mean,

            "Validation_Mean":
                validation_mean,

            "Gap":
                train_mean
                - validation_mean,
        })

    return pd.DataFrame(rows)

## 2. Logistic Regression - Baseline

In [ ]:
logistic_pipeline = Pipeline([
    (
        "preprocessor",
        build_scaled_preprocessor()
    ),

    (
        "classifier",
        LogisticRegression(
            random_state=RANDOM_STATE,
            max_iter=2000,
        )
    ),
])


logistic_cv_results = cross_validate(
    estimator=logistic_pipeline,
    X=X_train,
    y=y_train,
    cv=cv_strategy,
    scoring=scoring_metrics,
    return_train_score=True,
)


print(
    "Logistic Regression Baseline"
)

print(
    cv_summary(
        logistic_cv_results
    ).round(4)
)

print(
    "\nTrain / Validation:"
)

print(
    train_validation_summary(
        logistic_cv_results
    ).round(4)
)

Logistic Regression Baseline
      Metric    Mean     Std
0   ACCURACY  0.6099  0.0009
1  PRECISION  0.0000  0.0000
2     RECALL  0.0000  0.0000
3         F1  0.0000  0.0000
4    ROC_AUC  0.4774  0.0177

Train / Validation:
      Metric  Train_Mean  Validation_Mean     Gap
0   ACCURACY      0.6103           0.6099  0.0004
1  PRECISION      0.4000           0.0000  0.4000
2     RECALL      0.0010           0.0000  0.0010
3         F1      0.0020           0.0000  0.0020
4    ROC_AUC      0.5216           0.4774  0.0442


Logistic Regression baseline có xu hướng dự đoán hầu hết mẫu về lớp 0.

Do dữ liệu mất cân bằng và F1 của lớp Potable thấp,
mô hình tiếp tục được tuning bằng F1 Score.

## 3. Logistic Regression - Hyperparameter Tuning

In [ ]:
logistic_param_grid = {

    "classifier__C": [
        0.01,
        0.1,
        1.0,
        10.0,
        100.0,
    ],

    "classifier__class_weight": [
        None,
        "balanced",
    ],

    "classifier__solver": [
        "liblinear",
        "lbfgs",
    ],
}


logistic_grid_search = GridSearchCV(

    estimator=logistic_pipeline,

    param_grid=logistic_param_grid,

    scoring="f1",

    cv=cv_strategy,

    n_jobs=-1,

    verbose=0,

    return_train_score=True,
)


logistic_grid_search.fit(
    X_train,
    y_train
)


best_logistic_pipeline = (
    logistic_grid_search
    .best_estimator_
)


best_logistic_cv_results = (
    cross_validate(

        estimator=
            best_logistic_pipeline,

        X=X_train,

        y=y_train,

        cv=cv_strategy,

        scoring=
            scoring_metrics,

        return_train_score=True,
    )
)


print(
    "Best parameters:"
)

print(
    logistic_grid_search
    .best_params_
)


print(
    "\nBest CV F1:",
    round(
        logistic_grid_search
        .best_score_,
        4
    )
)


print(
    "\nCV sau tuning:"
)

print(
    cv_summary(
        best_logistic_cv_results
    ).round(4)
)


print(
    "\nTrain / Validation:"
)

print(
    train_validation_summary(
        best_logistic_cv_results
    ).round(4)
)

Best parameters:
{'classifier__C': 0.01, 'classifier__class_weight': 'balanced', 'classifier__solver': 'liblinear'}

Best CV F1: 0.4183

CV sau tuning:
      Metric    Mean     Std
0   ACCURACY  0.4962  0.0166
1  PRECISION  0.3802  0.0199
2     RECALL  0.4658  0.0456
3         F1  0.4183  0.0288
4    ROC_AUC  0.4766  0.0177

Train / Validation:
      Metric  Train_Mean  Validation_Mean     Gap
0   ACCURACY      0.5147           0.4962  0.0185
1  PRECISION      0.4021           0.3802  0.0218
2     RECALL      0.5012           0.4658  0.0354
3         F1      0.4462           0.4183  0.0279
4    ROC_AUC      0.5220           0.4766  0.0454


## 4. Support Vector Machine - Baseline

In [ ]:
svm_baseline_pipeline = Pipeline([
    (
        "preprocessor",
        build_scaled_preprocessor()
    ),

    (
        "classifier",
        SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
        )
    ),
])


svm_cv_results = cross_validate(

    estimator=
        svm_baseline_pipeline,

    X=X_train,

    y=y_train,

    cv=cv_strategy,

    scoring=
        scoring_metrics,

    return_train_score=True,
)


print(
    "SVM Baseline"
)

print(
    cv_summary(
        svm_cv_results
    ).round(4)
)


print(
    "\nTrain / Validation:"
)

print(
    train_validation_summary(
        svm_cv_results
    ).round(4)
)

SVM Baseline
      Metric    Mean     Std
0   ACCURACY  0.6798  0.0112
1  PRECISION  0.7190  0.0236
2     RECALL  0.2935  0.0320
3         F1  0.4161  0.0344
4    ROC_AUC  0.7019  0.0179

Train / Validation:
      Metric  Train_Mean  Validation_Mean     Gap
0   ACCURACY      0.7409           0.6798  0.0612
1  PRECISION      0.8683           0.7190  0.1493
2     RECALL      0.3960           0.2935  0.1025
3         F1      0.5438           0.4161  0.1277
4    ROC_AUC      0.8225           0.7019  0.1206


SVM baseline có Accuracy và ROC-AUC tốt hơn Logistic Regression.

Tuy nhiên Recall còn thấp, vì vậy tiếp tục tuning các tham số:
- C
- gamma
- class_weight

## 5. Support Vector Machine - Hyperparameter Tuning

In [ ]:
svm_tuning_pipeline = Pipeline([
    (
        "preprocessor",
        build_scaled_preprocessor()
    ),

    (
        "classifier",
        SVC(
            kernel="rbf"
        )
    ),
])


svm_param_grid = {

    "classifier__C": [
        0.1,
        1.0,
        10.0,
    ],

    "classifier__gamma": [
        "scale",
        0.01,
        0.1,
        1.0,
    ],

    "classifier__class_weight": [
        None,
        "balanced",
    ],
}


svm_grid_search = GridSearchCV(

    estimator=
        svm_tuning_pipeline,

    param_grid=
        svm_param_grid,

    scoring="f1",

    cv=cv_strategy,

    n_jobs=-1,

    verbose=0,

    return_train_score=True,
)


svm_grid_search.fit(
    X_train,
    y_train
)


best_svm_pipeline = (
    svm_grid_search
    .best_estimator_
)


best_svm_cv_results = (
    cross_validate(

        estimator=
            best_svm_pipeline,

        X=X_train,

        y=y_train,

        cv=cv_strategy,

        scoring=
            scoring_metrics,

        return_train_score=True,
    )
)


print(
    "Best parameters:"
)

print(
    svm_grid_search
    .best_params_
)


print(
    "\nBest CV F1:",
    round(
        svm_grid_search
        .best_score_,
        4
    )
)


print(
    "\nCV sau tuning:"
)

print(
    cv_summary(
        best_svm_cv_results
    ).round(4)
)


print(
    "\nTrain / Validation:"
)

print(
    train_validation_summary(
        best_svm_cv_results
    ).round(4)
)

Best parameters:
{'classifier__C': 1.0, 'classifier__class_weight': 'balanced', 'classifier__gamma': 'scale'}

Best CV F1: 0.5704

CV sau tuning:
      Metric    Mean     Std
0   ACCURACY  0.6668  0.0147
1  PRECISION  0.5733  0.0173
2     RECALL  0.5694  0.0523
3         F1  0.5704  0.0299
4    ROC_AUC  0.7036  0.0217

Train / Validation:
      Metric  Train_Mean  Validation_Mean     Gap
0   ACCURACY      0.7607           0.6668  0.0939
1  PRECISION      0.6893           0.5733  0.1160
2     RECALL      0.7038           0.5694  0.1343
3         F1      0.6964           0.5704  0.1261
4    ROC_AUC      0.8242           0.7036  0.1206


In [ ]:
training_comparison = pd.DataFrame([
    {
        "Model":
            "Logistic Baseline",

        "Accuracy":
            np.mean(
                logistic_cv_results[
                    "test_accuracy"
                ]
            ),

        "F1":
            np.mean(
                logistic_cv_results[
                    "test_f1"
                ]
            ),

        "ROC_AUC":
            np.mean(
                logistic_cv_results[
                    "test_roc_auc"
                ]
            ),
    },

    {
        "Model":
            "Logistic Tuned",

        "Accuracy":
            np.mean(
                best_logistic_cv_results[
                    "test_accuracy"
                ]
            ),

        "F1":
            np.mean(
                best_logistic_cv_results[
                    "test_f1"
                ]
            ),

        "ROC_AUC":
            np.mean(
                best_logistic_cv_results[
                    "test_roc_auc"
                ]
            ),
    },

    {
        "Model":
            "SVM Baseline",

        "Accuracy":
            np.mean(
                svm_cv_results[
                    "test_accuracy"
                ]
            ),

        "F1":
            np.mean(
                svm_cv_results[
                    "test_f1"
                ]
            ),

        "ROC_AUC":
            np.mean(
                svm_cv_results[
                    "test_roc_auc"
                ]
            ),
    },

    {
        "Model":
            "SVM Tuned",

        "Accuracy":
            np.mean(
                best_svm_cv_results[
                    "test_accuracy"
                ]
            ),

        "F1":
            np.mean(
                best_svm_cv_results[
                    "test_f1"
                ]
            ),

        "ROC_AUC":
            np.mean(
                best_svm_cv_results[
                    "test_roc_auc"
                ]
            ),
    },
])


print(
    training_comparison.round(4)
)

               Model  Accuracy      F1  ROC_AUC
0  Logistic Baseline    0.6099  0.0000   0.4774
1     Logistic Tuned    0.4962  0.4183   0.4766
2       SVM Baseline    0.6798  0.4161   0.7019
3          SVM Tuned    0.6668  0.5704   0.7036


## Kết luận Training

SVM sau tuning đạt kết quả Cross Validation tốt hơn
Logistic Regression, đặc biệt ở F1 và ROC-AUC.

Cấu hình SVM được chọn:

- kernel = rbf
- C = 1.0
- gamma = scale
- class_weight = balanced

Test set chưa được sử dụng trong quá trình tuning.
Việc đánh giá cuối cùng sẽ thực hiện trong `04_evaluate.ipynb`.